# XGBoost - Cross Validation (STROKE)

Notebook para executar XGBoost com StratifiedKFold sobre o dataset STROKE.
- Selecione qual dataset (raw ou normalizado) usar alterando a variável `DATASET_PATH` no bloco de configuração.
- Os parâmetros do modelo estão em `xgb_params`.
- Outputs: `model_reports/xgboost/cv/` contém `csv/`, `images/`, `pdf/`.
- O modelo será salvo em `databases/STROKE/common/models/xgboost/v1/xgb_model_cf{N_SPLITS}.pkl`. 

In [1]:
# Configuração
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE = Path('../../')  # ajustável se necessário
# Use o dataset raw de STROKE por padrão; EDA pode gerar versões normalizadas se desejar
# Nome do dataset cru: healthcare_stroke_data.csv
DATASET_NAME = 'healthcare_stroke_data_processed_cru_final.csv'  # ou stroke_standard_scaled.csv / stroke_robust_scaled.csv / stroke_minmax_scaled.csv
# Preferir carregar do raw; se quiser usar versões processadas, ajuste a linha abaixo
DATASET_PATH = Path('../data/processed') / DATASET_NAME
# Se quiser forçar explicitamente qual coluna é o target (no bruto ou no normalizado), defina aqui
# Para STROKE normalmente a coluna alvo é 'stroke' (0/1)
TARGET_COLUMN = 'stroke'  # ajustar se necessário

# Output paths
REPORTS = Path('../model_reports/xgboost/cv')
CSV_OUT = REPORTS / 'csv'
IMAGES_OUT = REPORTS / 'images'
PDF_OUT = REPORTS / 'pdf'
MODEL_DIR = Path('../common/models/xgboost/v1')
# Basico outputs (single train/test) - sibling folder to 'cv'
BASICO_DIR = REPORTS.parent / 'basico'
BASICO_CSV = BASICO_DIR / 'csv'
BASICO_IMAGES = BASICO_DIR / 'images'

for d in [REPORTS, CSV_OUT, IMAGES_OUT, PDF_OUT, MODEL_DIR, BASICO_DIR, BASICO_CSV, BASICO_IMAGES]:
    d.mkdir(parents=True, exist_ok=True)

# XGBoost params (variável conforme solicitado)
xgb_params = {
    "n_estimators": 650,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "gamma": 0.0,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "min_child_weight": 1,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "n_jobs": 8,
    "random_state": 42
}

# CV and threshold config
N_SPLITS = 10
THRESHOLD = 0.5
# Colunas identificadoras que NÃO devem entrar como features no modelo, mas devem ser mantidas nos arquivos de saída
EXCLUDE_COLUMNS = ['id', 'ID', 'patient_id', 'pid','y','stroke']

# Pareto configuration: threshold (fraction) and selected-features placeholder (set to None to be updated after CV)
PARETO_THRESHOLD = 0.9  # fraction, e.g. 0.8 == 80%
PARETO_SELECTED_FEATURES = None

# Variance percent-difference threshold (fraction). Example: 0.1 == 10% difference.
VAR_DIFF_PCT_THRESHOLD = 0.01
TEST_SIZE = 0.3
RT_RANDOM_STATE = 42

In [2]:
# Imports principais
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
                             balanced_accuracy_score, confusion_matrix, roc_curve, log_loss, auc)
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.auto import tqdm
import joblib

sns.set(style='whitegrid')

In [3]:
# Load dataset (path definido na configuração)
import shutil
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Arquivo de dataset não encontrado: {DATASET_PATH.resolve()}')

# Preserve a raw copy of the original CSV (unaltered) in data/processed with suffix _raw
RAW_COPY_PATH = Path('../data/processed') / f"{DATASET_NAME.replace('.csv','')}_raw.csv"
try:
    shutil.copy2(DATASET_PATH, RAW_COPY_PATH)
    print(f'Raw dataset copiado para: {RAW_COPY_PATH}')
except Exception as e_copy:
    print('Não foi possível copiar o CSV original para processed:', e_copy)

# Leia o dataset (continuamos a usar df para processamento posterior)
df = pd.read_csv(DATASET_PATH)
# Função utilitária para encontrar coluna aproximada (case/strip tolerant)
def find_column(df, name):
    # tenta match exato
    if name in df.columns:
        return name
    # tenta match case-insensitive
    low = {c.lower().strip(): c for c in df.columns}
    if name.lower().strip() in low:
        return low[name.lower().strip()]
    # tenta encontrar coluna que contenha o nome
    for c in df.columns:
        if name.lower().strip() in c.lower():
            return c
    return None

# Determina a coluna alvo: usa TARGET_COLUMN quando fornecida, senão infere
target_col = None
if TARGET_COLUMN:
    candidate = find_column(df, TARGET_COLUMN)
    if candidate is None:
        raise ValueError(f"TARGET_COLUMN='{TARGET_COLUMN}' não encontrada. Colunas: {list(df.columns)}")
    target_col = candidate
else:
    # heurística leve: nomes comuns
    for cand in ['y', 'diagnosis', 'target', 'label', 'class', 'diagnose']:
        candidate = find_column(df, cand)
        if candidate:
            target_col = candidate
            break
    # procura colunas binárias
    if target_col is None:
        for c in df.columns:
            if c.lower().strip() in ('id', 'patient_id', 'pid'):
                continue
            try:
                nunq = df[c].nunique(dropna=True)
            except Exception:
                nunq = 999
            if nunq == 2:
                target_col = c
                break
    # fallback para última coluna com poucos valores distintos
    if target_col is None:
        lastc = df.columns[-1]
        if df[lastc].nunique(dropna=True) <= 10:
            target_col = lastc
    if target_col is None:
        raise ValueError(f"Não foi possível inferir a coluna alvo. Defina TARGET_COLUMN ou verifique as colunas: {list(df.columns)}")

# Renomeia para 'y' e garante coluna disponível antes de acessar
if target_col != 'y':
    df = df.rename(columns={target_col: 'y'})
if 'y' not in df.columns:
    raise KeyError('Coluna y não encontrada após renomeação')

# Se y não for numérico, discretiza (M/B -> 1/0 ou factorize para múltiplas classes)
if not np.issubdtype(df['y'].dtype, np.number):
    unique_vals = list(df['y'].dropna().unique())
    low = [str(u).lower() for u in unique_vals]
    if set(low) <= set(['m', 'b']):
        mapping = {v: (1 if str(v).lower() == 'm' else 0) for v in unique_vals}
        df['y'] = df['y'].map(mapping)
    else:
        # factorize para inteiros 0..k-1
        df['y'], uniques = pd.factorize(df['y'])
    # salva versão discretizada para inspeção
    df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_y_numeric.csv', index=False)

# Substitui inf valores e dropna (pode ajustar imputação se preferir)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna()

# reorganiza para garantir que y seja a última coluna
cols = [c for c in df.columns if c != 'y'] + ['y']
df = df[cols]

# salva uma cópia de entrada no csv de reports (final)
df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}', index=False)

print('Dataset shape:', df.shape)

Raw dataset copiado para: ..\data\processed\healthcare_stroke_data_processed_cru_final_raw.csv
Dataset shape: (4259, 12)


In [10]:
# ==============================================================
# XGBOOST 8/1/1 NESTED-STYLE (Optuna TPE) — XGBoost 2.1.3 ok
# Ajustes principais:
# - Threshold NÃO otimiza por accuracy (padrão agora: F1)
# - Threshold search em faixa baixa (pois p_pos ~ 0.008–0.17)
# - Regra dura: threshold deve gerar TP>=1 na validação (evita F1=0)
# ==============================================================

import json, joblib, optuna
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
import xgboost as xgb

# -------------------- CONFIG --------------------
RANDOM_STATE      = 42
N_TRIALS          = 20
N_SPLITS_OUTER    = 10
N_SPLITS_INNER    = 8
EARLY_STOP_ROUNDS = 50
EPS               = 1e-12


OPTIMIZE_METRIC = "auc"
THRESH_OPTIMIZE_FOR = "f1"     # <<< principal
THRESH_MIN = 0.005
THRESH_MAX = 0.30
THRESH_GRID_STEP = 0.001
REQUIRE_TP_IN_VAL = True
MIN_TP_IN_VAL = 1


# Optuna: otimiza AUC (default) ou ACC
OPTIMIZE_METRIC = globals().get("OPTIMIZE_METRIC", "auc").lower().strip()
if OPTIMIZE_METRIC not in {"auc", "accuracy"}:
    raise ValueError("OPTIMIZE_METRIC deve ser 'auc' ou 'accuracy'.")

# >>>>> MUDE AQUI: threshold deve otimizar F1 ou BAC (NÃO accuracy)
THRESH_OPTIMIZE_FOR = globals().get("THRESH_OPTIMIZE_FOR", "f1").lower().strip()
if THRESH_OPTIMIZE_FOR not in {"accuracy", "f1", "bac"}:
    raise ValueError("THRESH_OPTIMIZE_FOR deve ser 'accuracy', 'f1' ou 'bac'.")

# >>>>> MUDE AQUI: faixa/step (suas probabilidades são baixas)
THRESH_GRID_STEP = float(globals().get("THRESH_GRID_STEP", 0.001))
THRESH_MIN = float(globals().get("THRESH_MIN", 0.005))
THRESH_MAX = float(globals().get("THRESH_MAX", 0.30))  # não precisa ir até 0.95 no seu caso

# Regra dura: exigir ao menos 1 TP na validação
REQUIRE_TP_IN_VAL = bool(globals().get("REQUIRE_TP_IN_VAL", True))
MIN_TP_IN_VAL = int(globals().get("MIN_TP_IN_VAL", 1))

# Saídas
OUT_DIR = Path("../model_reports/xgboost/cv811")
OUT_DIR.mkdir(parents=True, exist_ok=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"📦 XGBoost versão detectada: {xgb.__version__}")
print(f"🎯 Optuna otimizando por: {OPTIMIZE_METRIC.upper()}")
print(f"🎚️ Threshold otimiza: {THRESH_OPTIMIZE_FOR.upper()} | range=[{THRESH_MIN},{THRESH_MAX}] step={THRESH_GRID_STEP}")
print(f"🧷 REQUIRE_TP_IN_VAL={REQUIRE_TP_IN_VAL} | MIN_TP_IN_VAL={MIN_TP_IN_VAL}")

# -------------------- Dados e Features --------------------
try:
    df
except NameError:
    raise RuntimeError("df não está definido.")

try:
    EXCLUDE_COLUMNS
except NameError:
    EXCLUDE_COLUMNS = [
        '__cls_tmp__', 'orig_index', 'diagnosis', 'id', 'ID',
        'patient_id', 'pid', 'fold', 'y_train', 'prob_0', 'prob_1',
        'y_proba', 'y_pred'
    ]

y_raw = df["y"].copy()

exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
ID_COLS = [c for c in ID_COLS if c != "y"]
FEATURES = [c for c in df.columns if c not in ID_COLS + ["y"]]
X_raw = df[FEATURES].copy()

classes = np.unique(y_raw.dropna().values)
if len(classes) != 2:
    raise RuntimeError("Este script está configurado para classificação binária.")
class_to_bin = {classes[0]: 0, classes[1]: 1}
y = y_raw.map(class_to_bin).astype(int)

num_cols = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in FEATURES if c not in num_cols]

print(f"✅ Dados: X={X_raw.shape} | y={y.shape} | classes={classes} -> (0/1)")
print(f"✅ Colunas numéricas: {len(num_cols)} | categóricas: {len(cat_cols)}")

# -------------------- Preprocess --------------------
def fit_transform_fold(X_tr, X_va=None):
    if len(num_cols) > 0:
        imp_num = SimpleImputer(strategy="median")
        Xtr_num = imp_num.fit_transform(X_tr[num_cols])
        Xva_num = imp_num.transform(X_va[num_cols]) if X_va is not None else None
    else:
        imp_num = None
        Xtr_num = np.empty((len(X_tr), 0))
        Xva_num = np.empty((len(X_va), 0)) if X_va is not None else None

    if len(cat_cols) > 0:
        imp_cat = SimpleImputer(strategy="most_frequent")
        Xtr_cat_imp = imp_cat.fit_transform(X_tr[cat_cols])
        Xva_cat_imp = imp_cat.transform(X_va[cat_cols]) if X_va is not None else None

        enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, encoded_missing_value=-1)
        Xtr_cat = enc.fit_transform(Xtr_cat_imp)
        Xva_cat = enc.transform(Xva_cat_imp) if X_va is not None else None
    else:
        imp_cat = None
        enc = None
        Xtr_cat = np.empty((len(X_tr), 0))
        Xva_cat = np.empty((len(X_va), 0)) if X_va is not None else None

    X_tr_t = np.hstack([Xtr_num, Xtr_cat])
    X_va_t = np.hstack([Xva_num, Xva_cat]) if X_va is not None else None
    return X_tr_t, X_va_t, (imp_num, imp_cat, enc)

def make_xgb(params):
    return XGBClassifier(
        **params,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=0
    )

def _get_pos_proba(model, X_t):
    proba = model.predict_proba(X_t)
    return proba[:, 1] if proba.ndim == 2 else np.squeeze(proba)

def _fit_with_early_stopping(model, X_tr_t, y_tr, X_va_t, y_va):
    cb = []
    try:
        from xgboost.callback import EarlyStopping
        cb = [EarlyStopping(rounds=EARLY_STOP_ROUNDS, save_best=True, maximize=True)]
    except Exception:
        cb = []
    try:
        model.fit(X_tr_t, y_tr, eval_set=[(X_va_t, y_va)], verbose=False, callbacks=cb)
    except TypeError:
        model.fit(X_tr_t, y_tr)

# -------------------- Métricas --------------------
def cross_entropy_per_class(y_true, p_pos, eps=1e-12):
    p_pos = np.clip(np.asarray(p_pos), eps, 1 - eps)
    y_true = np.asarray(y_true).astype(int)
    mask0 = (y_true == 0)
    mask1 = (y_true == 1)
    ce0 = float(np.mean(-np.log(1 - p_pos[mask0]))) if np.any(mask0) else np.nan
    ce1 = float(np.mean(-np.log(p_pos[mask1])))     if np.any(mask1) else np.nan
    return ce0, ce1

def safe_auc(y_true, p_pos):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, p_pos))

def _bac_from_counts(TP, TN, FP, FN):
    rec1 = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    rec0 = TN / (TN + FP) if (TN + FP) > 0 else np.nan
    return float(np.nanmean([rec0, rec1]))

def metrics_for_table(y_true, proba, threshold=0.5, eps=1e-12):
    y_true = np.asarray(y_true).astype(int)
    proba  = np.asarray(proba)
    y_pred = (proba >= threshold).astype(int)

    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN

    acc  = float(accuracy_score(y_true, y_pred))
    f1   = float(f1_score(y_true, y_pred, zero_division=0))
    prec = float(precision_score(y_true, y_pred, zero_division=0))
    rec  = float(recall_score(y_true, y_pred, zero_division=0))
    bac  = _bac_from_counts(TP, TN, FP, FN)

    auc  = safe_auc(y_true, proba)
    ce0, ce1 = cross_entropy_per_class(y_true, proba, eps=eps)

    TP_pct = TP / total if total > 0 else np.nan
    FP_pct = FP / total if total > 0 else np.nan
    TN_pct = TN / total if total > 0 else np.nan
    FN_pct = FN / total if total > 0 else np.nan

    return {
        "ACC": acc, "F1": f1, "BAC": bac, "PRE": prec, "REC": rec, "AUC": auc,
        "CE-C0": ce0, "CE-C1": ce1, "CE-C2": "---", "y": 1,
        "TP%": TP_pct, "FP%": FP_pct, "TN%": TN_pct, "FN%": FN_pct
    }

# -------------------- Threshold com regra dura TP>0 na validação --------------------
def find_best_threshold_strict(y_true, p_pos, optimize_for="f1", step=0.001,
                               tmin=0.005, tmax=0.30, require_tp=True, min_tp=1):
    y_true = np.asarray(y_true).astype(int)
    p_pos  = np.asarray(p_pos)

    best_t = 0.5
    best_s = -np.inf
    best_aux_f1 = -np.inf
    best_aux_rec = -np.inf
    candidates = 0

    for t in np.arange(tmin, tmax + 1e-12, step):
        y_pred = (p_pos >= t).astype(int)

        TP = int(np.sum((y_true == 1) & (y_pred == 1)))
        TN = int(np.sum((y_true == 0) & (y_pred == 0)))
        FP = int(np.sum((y_true == 0) & (y_pred == 1)))
        FN = int(np.sum((y_true == 1) & (y_pred == 0)))

        if require_tp and TP < min_tp:
            continue

        candidates += 1

        if optimize_for == "accuracy":
            s = float(accuracy_score(y_true, y_pred))
        elif optimize_for == "f1":
            s = float(f1_score(y_true, y_pred, zero_division=0))
        else:  # bac
            s = _bac_from_counts(TP, TN, FP, FN)

        f1v = float(f1_score(y_true, y_pred, zero_division=0))
        recv = float(recall_score(y_true, y_pred, zero_division=0))

        # desempate: s > f1 > recall
        if (s > best_s) or (np.isclose(s, best_s) and f1v > best_aux_f1) or (np.isclose(s, best_s) and np.isclose(f1v, best_aux_f1) and recv > best_aux_rec):
            best_s = s
            best_aux_f1 = f1v
            best_aux_rec = recv
            best_t = float(t)

    used_fallback = False
    if candidates == 0:
        # fallback: escolhe um threshold bem baixo (quantil) para forçar positivos
        used_fallback = True
        best_t = float(np.quantile(p_pos, 0.90))  # top 10% vira positivo
        best_t = max(tmin, min(best_t, tmax))

    return best_t, best_s, used_fallback, candidates

# -------------------- Split 8/1/1 hold-out --------------------
outer_kf = StratifiedKFold(n_splits=N_SPLITS_OUTER, shuffle=True, random_state=RANDOM_STATE)
fold_indices = list(outer_kf.split(X_raw, y))

val1_idx  = fold_indices[-2][1]
test1_idx = fold_indices[-1][1]
train8_idx = np.setdiff1d(np.arange(len(X_raw)), np.concatenate([val1_idx, test1_idx]))

X_train8, y_train8 = X_raw.iloc[train8_idx], y.iloc[train8_idx]
X_val1,   y_val1   = X_raw.iloc[val1_idx],   y.iloc[val1_idx]
X_test1,  y_test1  = X_raw.iloc[test1_idx],  y.iloc[test1_idx]

print("\n=== Split 8/1/1 (hold-out) — CORRIGIDO ===")
print(f"Treino: {len(train8_idx)} | Val: {len(val1_idx)} | Teste: {len(test1_idx)}")

# -------------------- Optuna (CV interna) --------------------
inner_kf = StratifiedKFold(n_splits=N_SPLITS_INNER, shuffle=True, random_state=RANDOM_STATE)

def suggest_params(trial):
    return {
        "n_estimators":     trial.suggest_int("n_estimators", 150, 700),
        "max_depth":        trial.suggest_int("max_depth", 3, 12),
        "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 3e-1, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 12),
        "gamma":            trial.suggest_float("gamma", 0.0, 10.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "max_delta_step":   trial.suggest_int("max_delta_step", 0, 5),
    }

def objective(trial):
    params = suggest_params(trial)
    scores = []
    for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), start=1):
        X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
        y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

        X_tr_t, X_va_t, _ = fit_transform_fold(X_tr, X_va)
        model = make_xgb(params)
        _fit_with_early_stopping(model, X_tr_t, y_tr, X_va_t, y_va)

        p_pos = _get_pos_proba(model, X_va_t)

        if OPTIMIZE_METRIC == "auc":
            s = safe_auc(y_va, p_pos)
            if np.isnan(s):
                s = 0.5
        else:
            s = float(accuracy_score(y_va, (p_pos >= 0.5).astype(int)))

        scores.append(float(s))
        trial.report(float(np.mean(scores)), step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = dict(study.best_trial.params)
print(f"\n🏆 Melhor score ({OPTIMIZE_METRIC.upper()}): {study.best_value:.6f}")
print("🎯 Melhores hiperparâmetros:\n", json.dumps(best_params, indent=2))

# -------------------- Validação hold-out + threshold estrito --------------------
X_tr_t, X_val_t, _ = fit_transform_fold(X_train8, X_val1)
clf_val = make_xgb(best_params)
_fit_with_early_stopping(clf_val, X_tr_t, y_train8, X_val_t, y_val1)
p_val = _get_pos_proba(clf_val, X_val_t)

print("\n[DEBUG] Probabilidades na validação hold-out:")
print(f"min={float(np.min(p_val)):.6f} | p50={float(np.quantile(p_val,0.50)):.6f} | p90={float(np.quantile(p_val,0.90)):.6f} | max={float(np.max(p_val)):.6f}")

best_thr, best_thr_score, used_fb, cand = find_best_threshold_strict(
    y_true=y_val1, p_pos=p_val,
    optimize_for=THRESH_OPTIMIZE_FOR,
    step=THRESH_GRID_STEP,
    tmin=THRESH_MIN, tmax=THRESH_MAX,
    require_tp=REQUIRE_TP_IN_VAL, min_tp=MIN_TP_IN_VAL
)

print(f"\n🎚️ Threshold escolhido: {best_thr:.4f} | score({THRESH_OPTIMIZE_FOR})={best_thr_score:.6f} | candidatos={cand} | fallback={used_fb}")

# -------------------- Treino final + teste hold-out --------------------
X_train9 = pd.concat([X_train8, X_val1], axis=0)
y_train9 = pd.concat([y_train8, y_val1], axis=0)

X_tr9_t, X_te_t, _ = fit_transform_fold(X_train9, X_test1)
clf_final = make_xgb(best_params)
_fit_with_early_stopping(clf_final, X_tr9_t, y_train9, X_te_t, y_test1)
p_test = _get_pos_proba(clf_final, X_te_t)

print("\n[DEBUG] Probabilidades no teste hold-out:")
print(f"min={float(np.min(p_test)):.6f} | p50={float(np.quantile(p_test,0.50)):.6f} | p90={float(np.quantile(p_test,0.90)):.6f} | max={float(np.max(p_test)):.6f}")

# -------------------- Tabela final val/test --------------------
val_tab  = metrics_for_table(y_val1,  p_val,  threshold=best_thr, eps=EPS)
test_tab = metrics_for_table(y_test1, p_test, threshold=best_thr, eps=EPS)

df_final = pd.DataFrame([
    {"split": "validacao", "threshold": best_thr, **val_tab},
    {"split": "teste",     "threshold": best_thr, **test_tab},
])

cols_order = ["split","threshold","ACC","F1","BAC","PRE","REC","AUC","CE-C0","CE-C1","CE-C2","y","TP%","FP%","TN%","FN%"]
df_final = df_final[cols_order]

print("\n=== MÉTRICAS FINAIS (8/1/1) — threshold selecionado na validação ===")
print(df_final.to_string(index=False))

final_path = OUT_DIR / f"xgb_811_metrics_val_test_opt_{OPTIMIZE_METRIC}_thr_{THRESH_OPTIMIZE_FOR}_strictTP.csv"
df_final.to_csv(final_path, index=False)

# Persistência
joblib.dump(study, OUT_DIR / f"optuna_xgb_cv811_{OPTIMIZE_METRIC}_{N_TRIALS}trials.pkl")
with open(OUT_DIR / f"best_xgb_params_cv811_{OPTIMIZE_METRIC}.json", "w") as f:
    json.dump(best_params, f, indent=2)
joblib.dump(clf_final, OUT_DIR / f"xgb_model_cv811_final_opt_{OPTIMIZE_METRIC}.pkl")

print(f"\n✅ CSV final salvo em: {final_path}")
print(f"✅ Arquivos salvos em: {OUT_DIR.resolve()}")


📦 XGBoost versão detectada: 2.1.3
🎯 Optuna otimizando por: AUC
🎚️ Threshold otimiza: F1 | range=[0.005,0.3] step=0.001
🧷 REQUIRE_TP_IN_VAL=True | MIN_TP_IN_VAL=1
✅ Dados: X=(4259, 10) | y=(4259,) | classes=[0 1] -> (0/1)
✅ Colunas numéricas: 3 | categóricas: 7

=== Split 8/1/1 (hold-out) — CORRIGIDO ===
Treino: 3408 | Val: 426 | Teste: 425


Best trial: 16. Best value: 0.837092: 100%|██████████| 20/20 [00:24<00:00,  1.22s/it]



🏆 Melhor score (AUC): 0.837092
🎯 Melhores hiperparâmetros:
 {
  "n_estimators": 519,
  "max_depth": 9,
  "learning_rate": 0.007809090540276516,
  "subsample": 0.7820173088197031,
  "colsample_bytree": 0.882319405285726,
  "min_child_weight": 11,
  "gamma": 6.388704234048726,
  "reg_alpha": 0.001552166672845991,
  "reg_lambda": 2.6965647725340632e-05,
  "max_delta_step": 1
}

[DEBUG] Probabilidades na validação hold-out:
min=0.007543 | p50=0.011161 | p90=0.081256 | max=0.195118

🎚️ Threshold escolhido: 0.1420 | score(f1)=0.322581 | candidatos=173 | fallback=False

[DEBUG] Probabilidades no teste hold-out:
min=0.007051 | p50=0.013620 | p90=0.080656 | max=0.194641

=== MÉTRICAS FINAIS (8/1/1) — threshold selecionado na validação ===
    split  threshold      ACC       F1      BAC      PRE      REC      AUC    CE-C0    CE-C1 CE-C2  y      TP%      FP%      TN%      FN%
validacao      0.142 0.950704 0.322581 0.676569 0.277778 0.384615 0.857143 0.033729 2.563389   ---  1 0.011737 0.030516 0

In [ ]:
xgb_params

In [11]:
# ==============================================================
# XGBOOST 8/1/1 NESTED-STYLE (Optuna TPE) — compatível com versões antigas
# Ajustes:
# 1) Split 8/1/1 hold-out CORRETO (sem vazamento entre treino e teste)
# 2) Optuna otimiza por AUC ou ACC (parametrizável)
# 3) Threshold automático ROBUSTO: evita "tudo 0" e desempata com recall/F1
# 4) Saída final: print + CSV com ACC F1 BAC PRE REC AUC CE-C0 CE-C1 CE-C2 y TP% FP% TN% FN%
# ==============================================================

import json, joblib, optuna
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
import xgboost as xgb

# -------------------- CONFIG --------------------
RANDOM_STATE      = 42
N_TRIALS          = 20
N_SPLITS_OUTER    = 10
N_SPLITS_INNER    = 8
EARLY_STOP_ROUNDS = 50
EPS               = 1e-12

# Optuna: otimiza AUC (default) ou ACC
OPTIMIZE_METRIC = globals().get("OPTIMIZE_METRIC", "f1").lower().strip()
if OPTIMIZE_METRIC not in {"auc", "accuracy"}:
    raise ValueError("OPTIMIZE_METRIC deve ser 'auc' ou 'accuracy'.")

# Threshold automático: métrica principal (mas agora robusto)
THRESH_OPTIMIZE_FOR = globals().get("THRESH_OPTIMIZE_FOR", "f1").lower().strip()
if THRESH_OPTIMIZE_FOR not in {"accuracy", "f1", "bac"}:
    raise ValueError("THRESH_OPTIMIZE_FOR deve ser 'accuracy', 'f1' ou 'bac'.")

THRESH_GRID_STEP = float(globals().get("THRESH_GRID_STEP", 0.001))

# Limites para evitar thresholds extremos
THRESH_MIN = float(globals().get("THRESH_MIN", 0.005))
THRESH_MAX = float(globals().get("THRESH_MAX", 0.95))

# Exigir ao menos X positivos previstos na validação (evita F1=0 por "tudo 0")
# Pode ser inteiro (ex. 5) ou fração (ex. 0.01 => 1% do conjunto)
MIN_POS_PRED = globals().get("MIN_POS_PRED", 0.01)

# Saídas
OUT_DIR = Path("../model_reports/xgboost/cv811")
OUT_DIR.mkdir(parents=True, exist_ok=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"📦 XGBoost versão detectada: {xgb.__version__}")
print(f"🎯 Optuna otimizando por: {OPTIMIZE_METRIC.upper()}")
print(f"🎚️ Threshold robusto otimiza: {THRESH_OPTIMIZE_FOR.upper()} (validação hold-out)")
print(f"🔒 Threshold range: [{THRESH_MIN:.2f}, {THRESH_MAX:.2f}] | step={THRESH_GRID_STEP}")
print(f"🧷 MIN_POS_PRED: {MIN_POS_PRED} (fração ou inteiro)")

# -------------------- Dados e Features --------------------
try:
    df
except NameError:
    raise RuntimeError("df não está definido.")

try:
    EXCLUDE_COLUMNS
except NameError:
    EXCLUDE_COLUMNS = [
        '__cls_tmp__', 'orig_index', 'diagnosis', 'id', 'ID',
        'patient_id', 'pid', 'fold', 'y_train', 'prob_0', 'prob_1',
        'y_proba', 'y_pred'
    ]

y_raw = df["y"].copy()

exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
ID_COLS = [c for c in ID_COLS if c != "y"]
FEATURES = [c for c in df.columns if c not in ID_COLS + ["y"]]
X_raw = df[FEATURES].copy()

# garante binário 0/1
classes = np.unique(y_raw.dropna().values)
if len(classes) != 2:
    raise RuntimeError("Este script está configurado para classificação binária.")
class_to_bin = {classes[0]: 0, classes[1]: 1}
y = y_raw.map(class_to_bin).astype(int)

# Detecta tipos
num_cols = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in FEATURES if c not in num_cols]

print(f"✅ Dados: X={X_raw.shape} | y={y.shape} | classes={classes} -> (0/1)")
print(f"✅ Colunas numéricas: {len(num_cols)} | categóricas: {len(cat_cols)}")

# -------------------- Helpers de pré-processo e treino --------------------
def fit_transform_fold(X_tr, X_va=None):
    # numéricas
    if len(num_cols) > 0:
        imp_num = SimpleImputer(strategy="median")
        Xtr_num = imp_num.fit_transform(X_tr[num_cols])
        Xva_num = imp_num.transform(X_va[num_cols]) if X_va is not None else None
    else:
        imp_num = None
        Xtr_num = np.empty((len(X_tr), 0))
        Xva_num = np.empty((len(X_va), 0)) if X_va is not None else None

    # categóricas
    if len(cat_cols) > 0:
        imp_cat = SimpleImputer(strategy="most_frequent")
        Xtr_cat_imp = imp_cat.fit_transform(X_tr[cat_cols])
        Xva_cat_imp = imp_cat.transform(X_va[cat_cols]) if X_va is not None else None

        enc = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
            encoded_missing_value=-1
        )
        Xtr_cat = enc.fit_transform(Xtr_cat_imp)
        Xva_cat = enc.transform(Xva_cat_imp) if X_va is not None else None
    else:
        imp_cat = None
        enc = None
        Xtr_cat = np.empty((len(X_tr), 0))
        Xva_cat = np.empty((len(X_va), 0)) if X_va is not None else None

    X_tr_t = np.hstack([Xtr_num, Xtr_cat])
    X_va_t = np.hstack([Xva_num, Xva_cat]) if X_va is not None else None
    return X_tr_t, X_va_t, (imp_num, imp_cat, enc)

def make_xgb(params):
    return XGBClassifier(
        **params,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=0
    )

def _get_pos_proba(model, X_t):
    proba = model.predict_proba(X_t)
    return proba[:, 1] if proba.ndim == 2 else np.squeeze(proba)

def _fit_with_early_stopping(model, X_tr_t, y_tr, X_va_t, y_va):
    cb = []
    try:
        from xgboost.callback import EarlyStopping
        cb = [EarlyStopping(rounds=EARLY_STOP_ROUNDS, save_best=True, maximize=True)]
    except Exception:
        cb = []
    try:
        model.fit(X_tr_t, y_tr, eval_set=[(X_va_t, y_va)], verbose=False, callbacks=cb)
    except TypeError:
        model.fit(X_tr_t, y_tr)

# -------------------- Métricas --------------------
def cross_entropy_per_class(y_true, p_pos, eps=1e-12):
    p_pos = np.clip(np.asarray(p_pos), eps, 1 - eps)
    y_true = np.asarray(y_true).astype(int)
    mask0 = (y_true == 0)
    mask1 = (y_true == 1)
    ce0 = float(np.mean(-np.log(1 - p_pos[mask0]))) if np.any(mask0) else np.nan
    ce1 = float(np.mean(-np.log(p_pos[mask1])))     if np.any(mask1) else np.nan
    return ce0, ce1

def safe_auc(y_true, p_pos):
    y_true = np.asarray(y_true).astype(int)
    if len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, p_pos))

def _bac_from_counts(TP, TN, FP, FN):
    rec1 = TP / (TP + FN) if (TP + FN) > 0 else np.nan
    rec0 = TN / (TN + FP) if (TN + FP) > 0 else np.nan
    return float(np.nanmean([rec0, rec1]))

def metrics_for_table(y_true, proba, threshold=0.5, eps=1e-12):
    y_true = np.asarray(y_true).astype(int)
    proba  = np.asarray(proba)
    y_pred = (proba >= threshold).astype(int)

    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN

    acc  = float(accuracy_score(y_true, y_pred))
    f1   = float(f1_score(y_true, y_pred, zero_division=0))
    prec = float(precision_score(y_true, y_pred, zero_division=0))
    rec  = float(recall_score(y_true, y_pred, zero_division=0))
    bac  = _bac_from_counts(TP, TN, FP, FN)

    auc  = safe_auc(y_true, proba)
    ce0, ce1 = cross_entropy_per_class(y_true, proba, eps=eps)

    TP_pct = TP / total if total > 0 else np.nan
    FP_pct = FP / total if total > 0 else np.nan
    TN_pct = TN / total if total > 0 else np.nan
    FN_pct = FN / total if total > 0 else np.nan

    return {
        "ACC": acc, "F1": f1, "BAC": bac, "PRE": prec, "REC": rec, "AUC": auc,
        "CE-C0": ce0, "CE-C1": ce1, "CE-C2": "---", "y": 1,
        "TP%": TP_pct, "FP%": FP_pct, "TN%": TN_pct, "FN%": FN_pct
    }

def all_metrics_table_long(y_true, proba, threshold=0.5, eps=1e-12):
    y_true = np.asarray(y_true).astype(int)
    proba  = np.asarray(proba)
    y_pred = (proba >= threshold).astype(int)

    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN

    n0 = int(np.sum(y_true == 0))
    n1 = int(np.sum(y_true == 1))
    prop_c0 = n0 / total if total > 0 else np.nan
    prop_c1 = n1 / total if total > 0 else np.nan

    ce0, ce1 = cross_entropy_per_class(y_true, proba, eps=eps)

    acc  = float(accuracy_score(y_true, y_pred))
    f1   = float(f1_score(y_true, y_pred, zero_division=0))
    prec = float(precision_score(y_true, y_pred, zero_division=0))
    rec  = float(recall_score(y_true, y_pred, zero_division=0))
    bac  = _bac_from_counts(TP, TN, FP, FN)

    auc  = safe_auc(y_true, proba)

    TP_pct = TP / total if total > 0 else np.nan
    FP_pct = FP / total if total > 0 else np.nan
    TN_pct = TN / total if total > 0 else np.nan
    FN_pct = FN / total if total > 0 else np.nan

    return {
        "Acurácia": acc,
        "Cross-Entropy C0": ce0,
        "Proporção C0": prop_c0,
        "Cross-Entropy C1": ce1,
        "Proporção C1": prop_c1,
        "F1 Score": f1,
        "Balanced Accuracy": bac,
        "Precision": prec,
        "Recall": rec,
        "AUROC": auc,
        "TP": TP, "FP": FP, "TN": TN, "FN": FN,
        "TP(%)": TP_pct, "FP(%)": FP_pct, "TN(%)": TN_pct, "FN(%)": FN_pct,
    }

# -------------------- Threshold ROBUSTO (evita "tudo 0") --------------------
def _min_pos_pred_as_int(n, min_pos_pred):
    if isinstance(min_pos_pred, float):
        if min_pos_pred <= 0:
            return 0
        return int(np.ceil(n * min_pos_pred))
    try:
        return int(min_pos_pred)
    except Exception:
        return 0

def find_best_threshold_robust(y_true, p_pos, optimize_for="accuracy", step=0.01,
                               tmin=0.05, tmax=0.95, min_pos_pred=0.01):
    """
    Seleção robusta de threshold:
    - busca apenas em [tmin, tmax] (evita extremos)
    - descarta thresholds que gerem poucos positivos previstos (min_pos_pred)
    - em empate: prefere maior F1 e maior Recall (para evitar F1=0)
    - fallback: se TODOS descartados, usa 0.5 e avisa via retorno
    """
    y_true = np.asarray(y_true).astype(int)
    p_pos  = np.asarray(p_pos)

    n = len(y_true)
    min_pos_int = _min_pos_pred_as_int(n, min_pos_pred)

    best_t = 0.5
    best_primary = -np.inf
    best_f1 = -np.inf
    best_rec = -np.inf
    used_fallback = False

    thresholds = np.arange(tmin, tmax + 1e-9, step)
    candidates_found = 0

    for t in thresholds:
        y_pred = (p_pos >= t).astype(int)
        pos_pred = int(np.sum(y_pred == 1))
        if pos_pred < max(1, min_pos_int):
            continue  # evita "tudo 0" (ou quase tudo 0)

        candidates_found += 1

        TP = int(np.sum((y_true == 1) & (y_pred == 1)))
        TN = int(np.sum((y_true == 0) & (y_pred == 0)))
        FP = int(np.sum((y_true == 0) & (y_pred == 1)))
        FN = int(np.sum((y_true == 1) & (y_pred == 0)))

        if optimize_for == "accuracy":
            primary = float(accuracy_score(y_true, y_pred))
        elif optimize_for == "f1":
            primary = float(f1_score(y_true, y_pred, zero_division=0))
        else:  # bac
            primary = _bac_from_counts(TP, TN, FP, FN)

        f1v = float(f1_score(y_true, y_pred, zero_division=0))
        recv = float(recall_score(y_true, y_pred, zero_division=0))

        # desempate robusto: primary > f1 > recall
        if (primary > best_primary) or \
           (np.isclose(primary, best_primary) and f1v > best_f1) or \
           (np.isclose(primary, best_primary) and np.isclose(f1v, best_f1) and recv > best_rec):
            best_primary = primary
            best_f1 = f1v
            best_rec = recv
            best_t = float(t)

    if candidates_found == 0:
        used_fallback = True
        best_t = 0.5
        best_primary = float(accuracy_score(y_true, (p_pos >= 0.5).astype(int))) if optimize_for == "accuracy" else \
                       float(f1_score(y_true, (p_pos >= 0.5).astype(int), zero_division=0)) if optimize_for == "f1" else \
                       _bac_from_counts(
                           int(np.sum((y_true == 1) & ((p_pos >= 0.5).astype(int) == 1))),
                           int(np.sum((y_true == 0) & ((p_pos >= 0.5).astype(int) == 0))),
                           int(np.sum((y_true == 0) & ((p_pos >= 0.5).astype(int) == 1))),
                           int(np.sum((y_true == 1) & ((p_pos >= 0.5).astype(int) == 0))),
                       )

    return best_t, best_primary, used_fallback, candidates_found

# -------------------- Folds externos (8/1/1) CORRIGIDO --------------------
outer_kf = StratifiedKFold(n_splits=N_SPLITS_OUTER, shuffle=True, random_state=RANDOM_STATE)
fold_indices = list(outer_kf.split(X_raw, y))

val1_idx  = fold_indices[-2][1]  # fold 9 (hold-out validação)
test1_idx = fold_indices[-1][1]  # fold 10 (hold-out teste)
train8_idx = np.setdiff1d(np.arange(len(X_raw)), np.concatenate([val1_idx, test1_idx]))

X_train8, y_train8 = X_raw.iloc[train8_idx], y.iloc[train8_idx]
X_val1,   y_val1   = X_raw.iloc[val1_idx],   y.iloc[val1_idx]
X_test1,  y_test1  = X_raw.iloc[test1_idx],  y.iloc[test1_idx]

print("\n=== Split 8/1/1 (hold-out) — CORRIGIDO ===")
print(f"Treino (8/10): {len(train8_idx)} | Validação hold-out (1/10): {len(val1_idx)} | Teste hold-out (1/10): {len(test1_idx)}")
print("✅ Validação e Teste NÃO entram no Optuna.\n")

# -------------------- Optuna (CV interna 8 folds) --------------------
inner_kf = StratifiedKFold(n_splits=N_SPLITS_INNER, shuffle=True, random_state=RANDOM_STATE)

def suggest_params(trial):
    return {
        "n_estimators":     trial.suggest_int("n_estimators", 150, 700),
        "max_depth":        trial.suggest_int("max_depth", 3, 12),
        "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 3e-1, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.3, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 12),
        "gamma":            trial.suggest_float("gamma", 0.0, 10.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "max_delta_step":   trial.suggest_int("max_delta_step", 0, 5),
    }

def objective(trial):
    params = suggest_params(trial)
    scores = []

    for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), start=1):
        X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
        y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

        X_tr_t, X_va_t, _ = fit_transform_fold(X_tr, X_va)
        model = make_xgb(params)
        _fit_with_early_stopping(model, X_tr_t, y_tr, X_va_t, y_va)

        p_pos = _get_pos_proba(model, X_va_t)

        if OPTIMIZE_METRIC == "auc":
            s = safe_auc(y_va, p_pos)
            if np.isnan(s):
                s = 0.5
        else:
            y_hat = (p_pos >= 0.5).astype(int)
            s = float(accuracy_score(y_va, y_hat))

        scores.append(float(s))
        trial.report(float(np.mean(scores)), step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(scores))

sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True, group=True, n_startup_trials=8)
pruner  = optuna.pruners.MedianPruner(n_warmup_steps=2)

study = optuna.create_study(
    study_name=f"xgb_optuna_cv811_{OPTIMIZE_METRIC}_catnum",
    direction="maximize",
    sampler=sampler,
    pruner=pruner
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_trial
best_params = dict(best.params)
print(f"\n🏆 Melhor score ({OPTIMIZE_METRIC.upper()}, CV interna 8 folds): {best.value:.6f}")
print("🎯 Melhores hiperparâmetros:\n", json.dumps(best_params, indent=2))

# -------------------- (1) Reavalia inner CV com best_params --------------------
inner_fold_rows = []
for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), start=1):
    X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
    y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

    X_tr_t, X_va_t, _ = fit_transform_fold(X_tr, X_va)
    model = make_xgb(best_params)
    _fit_with_early_stopping(model, X_tr_t, y_tr, X_va_t, y_va)

    p_pos = _get_pos_proba(model, X_va_t)
    metrics = all_metrics_table_long(y_va, p_pos, threshold=0.5, eps=EPS)  # 0.5 no inner (comparável)
    inner_fold_rows.append({"etapa": "inner_cv", "fold": fold_id, **metrics})

df_inner = pd.DataFrame(inner_fold_rows)
df_inner.to_csv(OUT_DIR / "cv_811_inner_folds.csv", index=False)

# -------------------- (2) Validação hold-out (fold 9) + threshold robusto --------------------
X_tr_t, X_val_t, _ = fit_transform_fold(X_train8, X_val1)
clf_val = make_xgb(best_params)
_fit_with_early_stopping(clf_val, X_tr_t, y_train8, X_val_t, y_val1)

p_val = _get_pos_proba(clf_val, X_val_t)

# DEBUG útil: range das probabilidades
print("\n[DEBUG] Probabilidades na validação hold-out:")
print(f"min={float(np.min(p_val)):.6f} | p10={float(np.quantile(p_val,0.10)):.6f} | "
      f"p50={float(np.quantile(p_val,0.50)):.6f} | p90={float(np.quantile(p_val,0.90)):.6f} | max={float(np.max(p_val)):.6f}")

best_thr, best_thr_score, used_fallback, n_cand = find_best_threshold_robust(
    y_true=y_val1, p_pos=p_val,
    optimize_for=THRESH_OPTIMIZE_FOR,
    step=THRESH_GRID_STEP,
    tmin=THRESH_MIN, tmax=THRESH_MAX,
    min_pos_pred=MIN_POS_PRED
)

msg_fb = " (FALLBACK 0.5)" if used_fallback else ""
print(f"\n🎚️ Threshold ótimo ROBUSTO{msg_fb}: {best_thr:.3f} | score({THRESH_OPTIMIZE_FOR})={best_thr_score:.6f} | candidatos válidos={n_cand}")

val_metrics_long = all_metrics_table_long(y_val1, p_val, threshold=best_thr, eps=EPS)
df_val = pd.DataFrame([{"etapa": "validacao", "fold": 9, **val_metrics_long}])
df_val.to_csv(OUT_DIR / "cv_811_validation.csv", index=False)

# -------------------- (3) Treino final (8+1) e teste hold-out (fold 10) com threshold robusto --------------------
X_train9 = pd.concat([X_train8, X_val1], axis=0)
y_train9 = pd.concat([y_train8, y_val1], axis=0)

X_tr9_t, X_te_t, _ = fit_transform_fold(X_train9, X_test1)
clf_final = make_xgb(best_params)
_fit_with_early_stopping(clf_final, X_tr9_t, y_train9, X_te_t, y_test1)

p_test = _get_pos_proba(clf_final, X_te_t)

# DEBUG útil: range das probabilidades no teste
print("\n[DEBUG] Probabilidades no teste hold-out:")
print(f"min={float(np.min(p_test)):.6f} | p10={float(np.quantile(p_test,0.10)):.6f} | "
      f"p50={float(np.quantile(p_test,0.50)):.6f} | p90={float(np.quantile(p_test,0.90)):.6f} | max={float(np.max(p_test)):.6f}")

test_metrics_long = all_metrics_table_long(y_test1, p_test, threshold=best_thr, eps=EPS)
df_test = pd.DataFrame([{"etapa": "teste", "fold": 10, **test_metrics_long}])
df_test.to_csv(OUT_DIR / "cv_811_test.csv", index=False)

# -------------------- (4) Consolidação (formato longo) --------------------
df_all = pd.concat([df_inner, df_val, df_test], ignore_index=True)
df_all.to_csv(OUT_DIR / "cv_811_results_long.csv", index=False)

# -------------------- (5) Tabela final (validação + teste) — formato dissertação --------------------
val_tab  = metrics_for_table(y_val1,  p_val,  threshold=best_thr, eps=EPS)
test_tab = metrics_for_table(y_test1, p_test, threshold=best_thr, eps=EPS)

df_final = pd.DataFrame([
    {"split": "validacao", "threshold": best_thr, **val_tab},
    {"split": "teste",     "threshold": best_thr, **test_tab},
])

cols_order = ["split","threshold","ACC","F1","BAC","PRE","REC","AUC","CE-C0","CE-C1","CE-C2","y","TP%","FP%","TN%","FN%"]
df_final = df_final[cols_order]

print("\n=== MÉTRICAS FINAIS (8/1/1) — threshold robusto (evita F1=0 por 'tudo 0') ===")
print(df_final.to_string(index=False))

final_path = OUT_DIR / f"xgb_811_metrics_val_test_opt_{OPTIMIZE_METRIC}_thr_{THRESH_OPTIMIZE_FOR}_robust.csv"
df_final.to_csv(final_path, index=False)

# -------------------- Persistência --------------------
joblib.dump(study, OUT_DIR / f"optuna_xgb_cv811_{OPTIMIZE_METRIC}_{N_TRIALS}trials.pkl")
with open(OUT_DIR / f"best_xgb_params_cv811_{OPTIMIZE_METRIC}.json", "w") as f:
    json.dump(best_params, f, indent=2)
joblib.dump(clf_final, OUT_DIR / f"xgb_model_cv811_final_opt_{OPTIMIZE_METRIC}.pkl")

# salva threshold e configs (pra reproduzir)
thr_info = {
    "threshold": best_thr,
    "optimize_metric": OPTIMIZE_METRIC,
    "thresh_optimize_for": THRESH_OPTIMIZE_FOR,
    "thresh_grid_step": THRESH_GRID_STEP,
    "thresh_min": THRESH_MIN,
    "thresh_max": THRESH_MAX,
    "min_pos_pred": MIN_POS_PRED,
    "used_fallback": used_fallback,
    "candidates_valid": n_cand
}
with open(OUT_DIR / f"best_threshold_cv811_{THRESH_OPTIMIZE_FOR}_robust.json", "w") as f:
    json.dump(thr_info, f, indent=2)

print(f"\n✅ CSV final salvo em: {final_path}")
print(f"✅ Threshold salvo em: {OUT_DIR / f'best_threshold_cv811_{THRESH_OPTIMIZE_FOR}_robust.json'}")
print(f"✅ Arquivos salvos em: {OUT_DIR.resolve()}")


📦 XGBoost versão detectada: 2.1.3
🎯 Optuna otimizando por: AUC
🎚️ Threshold robusto otimiza: F1 (validação hold-out)
🔒 Threshold range: [0.01, 0.30] | step=0.001
🧷 MIN_POS_PRED: 0.01 (fração ou inteiro)
✅ Dados: X=(4259, 10) | y=(4259,) | classes=[0 1] -> (0/1)
✅ Colunas numéricas: 3 | categóricas: 7

=== Split 8/1/1 (hold-out) — CORRIGIDO ===
Treino (8/10): 3408 | Validação hold-out (1/10): 426 | Teste hold-out (1/10): 425
✅ Validação e Teste NÃO entram no Optuna.



Best trial: 18. Best value: 0.836446: 100%|██████████| 20/20 [00:28<00:00,  1.44s/it]



🏆 Melhor score (AUC, CV interna 8 folds): 0.836446
🎯 Melhores hiperparâmetros:
 {
  "n_estimators": 458,
  "max_depth": 6,
  "learning_rate": 0.011243511393056016,
  "subsample": 0.5176639256623752,
  "colsample_bytree": 0.529185466587248,
  "min_child_weight": 2,
  "gamma": 7.83472920678917,
  "reg_alpha": 0.08000179829424227,
  "reg_lambda": 3.86984276848558e-08,
  "max_delta_step": 0
}

[DEBUG] Probabilidades na validação hold-out:
min=0.009042 | p10=0.009124 | p50=0.012671 | p90=0.085316 | max=0.234975

🎚️ Threshold ótimo ROBUSTO: 0.106 | score(f1)=0.311111 | candidatos válidos=188

[DEBUG] Probabilidades no teste hold-out:
min=0.008336 | p10=0.008402 | p50=0.013621 | p90=0.080450 | max=0.174608

=== MÉTRICAS FINAIS (8/1/1) — threshold robusto (evita F1=0 por 'tudo 0') ===
    split  threshold      ACC       F1      BAC      PRE      REC      AUC    CE-C0    CE-C1 CE-C2  y      TP%      FP%      TN%      FN%
validacao      0.106 0.927230 0.311111 0.738964 0.218750 0.538462 0.84634

In [ ]:
# # Diagnóstico rápido para NaN / tipos em train_df e test_df
# for name, ycol in [('train_df', 'y_train'), ('test_df', 'y_test')]:
#     df_block = globals().get(name)
#     if df_block is None:
#         print(f"{name} não existe")
#         continue
#     if df_block.empty:
#         print(f"{name} existe mas está vazio")
#         continue
#     print(f"\n{name}: shape={df_block.shape}")
#     for col in [ycol, 'prob_1', 'y_proba']:
#         if col in df_block.columns:
#             print(f"  {col}: nulls={df_block[col].isnull().sum()}, dtype={df_block[col].dtype}, unique_count={df_block[col].nunique(dropna=True)}")
#     cols_to_show = [c for c in [ycol, 'prob_1', 'y_proba'] if c in df_block.columns]
#     display(df_block[cols_to_show].head(10))

In [ ]:
# ===================== RECONSTRUIR train_df / test_df (se possível) =====================
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import roc_curve, auc, confusion_matrix

# Pastas auxiliares
try:
    IMAGES_OUT
except NameError:
    IMAGES_OUT = PDF_OUT.parent / "images"
IMAGES_OUT.mkdir(parents=True, exist_ok=True)

def _safe_concat(lst):
    return pd.concat(lst, ignore_index=True) if (isinstance(lst, list) and len(lst) > 0) else pd.DataFrame()

# Tenta reconstruir os dataframes globais a partir das partes do CV
X_train_global = _safe_concat(globals().get('X_train_parts', []))
y_train_global = _safe_concat(globals().get('y_train_parts', []))
y_train_pred_global = _safe_concat(globals().get('y_train_pred_parts', []))

X_test_global  = _safe_concat(globals().get('X_test_parts', []))
y_test_global  = _safe_concat(globals().get('y_test_parts', []))
y_test_pred_global  = _safe_concat(globals().get('y_test_pred_parts', []))

# Renomeia coluna "index" para "orig_index" (índice original do df)
for _df in [X_train_global, y_train_global, y_train_pred_global,
            X_test_global,  y_test_global,  y_test_pred_global]:
    if isinstance(_df, pd.DataFrame) and ('index' in _df.columns):
        _df.rename(columns={'index': 'orig_index'}, inplace=True)

# Monta train_df e test_df completos (com y_true, prob_1, y_pred)
if not X_train_global.empty and not y_train_global.empty:
    train_df = X_train_global.merge(y_train_global, on=['orig_index','fold'], how='left')
    if not y_train_pred_global.empty:
        train_df = train_df.merge(y_train_pred_global, on=['orig_index','fold'], how='left')
    if 'prob_1' in train_df.columns:
        train_df['y_proba'] = train_df['prob_1']
        train_df['y_pred'] = (train_df['y_proba'] >= THRESHOLD).astype(int)
else:
    train_df = pd.DataFrame()

if not X_test_global.empty and not y_test_global.empty:
    test_df = X_test_global.merge(y_test_global, on=['orig_index','fold'], how='left')
    if not y_test_pred_global.empty:
        test_df = test_df.merge(y_test_pred_global, on=['orig_index','fold'], how='left')
    if 'prob_1' in test_df.columns:
        test_df['y_proba'] = test_df['prob_1']
        test_df['y_pred'] = (test_df['y_proba'] >= THRESHOLD).astype(int)
else:
    test_df = pd.DataFrame()

# ===================== PDF RESUMO XGBOOST =====================
with PdfPages(PDF_OUT / f'XGB_CV_SUMMARY_{N_SPLITS}.pdf') as pdf:
    # -------- Página de parâmetros --------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    parametros = xgb_params.copy()
    parametros['threshold'] = THRESHOLD
    parametros['n_splits'] = N_SPLITS
    texto = 'Algoritmo: XGBoost\n' + '\n'.join([f'{k}: {v}' for k, v in parametros.items()])
    ax.text(0, 1, texto, fontsize=11, family='monospace', va='top')
    ax.set_title('📌 Parâmetros do Modelo - XGBoost')
    try:
        png_path = IMAGES_OUT / f'params_{N_SPLITS}.png'
        fig.savefig(png_path, dpi=300, bbox_inches='tight')
    except Exception:
        pass
    pdf.savefig(fig); plt.close()

    # -------- Página de métricas resumidas --------
    if 'df_resultados' in globals() and not df_resultados.empty:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.axis('off')
        display_df = df_resultados.groupby(['Conjunto']).mean(numeric_only=True).T.round(4)
        table = ax.table(cellText=display_df.values,
                         colLabels=display_df.columns,
                         rowLabels=display_df.index,
                         loc='center')
        table.auto_set_font_size(False); table.set_fontsize(9)
        try:
            png_path = IMAGES_OUT / f'metrics_summary_{N_SPLITS}.png'
            fig.savefig(png_path, dpi=300, bbox_inches='tight')
        except Exception:
            pass
        pdf.savefig(fig); plt.close()

    # -------- ROC - Treino vs Teste --------
    try:
        plotted = False
        # Preferência: usar dataframes globais
        if ('y_train' in train_df.columns) and ('y_proba' in train_df.columns):
            y_train_all = train_df['y_train']; y_score_train_all = train_df['y_proba']
            fpr_train, tpr_train, _ = roc_curve(y_train_all, y_score_train_all)
            roc_auc_train = auc(fpr_train, tpr_train); plotted = True
        elif 'fprs_train' in globals() and len(fprs_train) > 0:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = [np.interp(mean_fpr, fpr, tpr) for fpr, tpr in zip(fprs_train, tprs_train)]
            for t in tprs: t[0] = 0.0
            mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
            fpr_train, tpr_train = mean_fpr, mean_tpr
            roc_auc_train = np.mean(aurocs_train) if 'aurocs_train' in globals() and len(aurocs_train)>0 else auc(fpr_train,tpr_train)
            plotted = True

        if ('y_test' in test_df.columns) and ('y_proba' in test_df.columns):
            y_test_all = test_df['y_test']; y_score_test_all = test_df['y_proba']
            fpr_test, tpr_test, _ = roc_curve(y_test_all, y_score_test_all)
            roc_auc_test = auc(fpr_test, tpr_test); plotted = True
        elif 'fprs_test' in globals() and len(fprs_test) > 0:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = [np.interp(mean_fpr, fpr, tpr) for fpr, tpr in zip(fprs_test, tprs_test)]
            for t in tprs: t[0] = 0.0
            mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
            fpr_test, tpr_test = mean_fpr, mean_tpr
            roc_auc_test = np.mean(aurocs_test) if 'aurocs_test' in globals() and len(aurocs_test)>0 else auc(fpr_test,tpr_test)
            plotted = True

        if plotted:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            try:
                axes[0].plot(fpr_train, tpr_train, label=f'AUC = {roc_auc_train:.4f}')
                axes[0].plot([0,1],[0,1],'k--',lw=1)
                axes[0].set_title('ROC - Treino'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
                axes[0].legend(loc='lower right')
            except Exception:
                axes[0].text(0.5,0.5,'Sem dados de treino',ha='center'); axes[0].axis('off')
            try:
                axes[1].plot(fpr_test, tpr_test, label=f'AUC = {roc_auc_test:.4f}')
                axes[1].plot([0,1],[0,1],'k--',lw=1)
                axes[1].set_title('ROC - Teste'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
                axes[1].legend(loc='lower right')
            except Exception:
                axes[1].text(0.5,0.5,'Sem dados de teste',ha='center'); axes[1].axis('off')
            plt.tight_layout()
            try:
                png_path = IMAGES_OUT / f'roc_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()
        else:
            print('Não foi possível gerar curvas ROC (dados ausentes)')
    except Exception as e:
        print('Não foi possível gerar ROC agregado:', e)

    # -------- Matrizes de Confusão --------
    try:
        def build_cm_from_df(df_block, true_col, pred_col):
            y_true = df_block[true_col]; y_pred = df_block[pred_col]
            cm = confusion_matrix(y_true, y_pred)
            total = cm.sum(); cm_percent = cm/total*100 if total>0 else np.zeros_like(cm, dtype=float)
            return cm, cm_percent

        cm_train = cm_train_pct = cm_test = cm_test_pct = None

        if ('y_train' in train_df.columns) and ('y_pred' in train_df.columns):
            cm_train, cm_train_pct = build_cm_from_df(train_df, 'y_train', 'y_pred')
        elif ('df_resultados' in globals()) and ('TP' in df_resultados.columns):
            tr = df_resultados[df_resultados['Conjunto']=='Treino']
            TP,FP,TN,FN = int(tr['TP'].sum()), int(tr['FP'].sum()), int(tr['TN'].sum()), int(tr['FN'].sum())
            cm_train = np.array([[TN, FP],[FN, TP]])
            total = cm_train.sum(); cm_train_pct = cm_train/total*100 if total>0 else np.zeros_like(cm_train,float)

        if ('y_test' in test_df.columns) and ('y_pred' in test_df.columns):
            cm_test, cm_test_pct = build_cm_from_df(test_df, 'y_test', 'y_pred')
        elif ('df_resultados' in globals()) and ('TP' in df_resultados.columns):
            ts = df_resultados[df_resultados['Conjunto']=='Teste']
            TP,FP,TN,FN = int(ts['TP'].sum()), int(ts['FP'].sum()), int(ts['TN'].sum()), int(ts['FN'].sum())
            cm_test = np.array([[TN, FP],[FN, TP]])
            total = cm_test.sum(); cm_test_pct = cm_test/total*100 if total>0 else np.zeros_like(cm_test,float)

        if (cm_train is None and cm_test is None):
            print('Não há dados suficientes para gerar matrizes de confusão.')
        else:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            def plot_cm(ax, cm, cm_pct, title):
                if cm is None:
                    ax.text(0.5,0.5,'Sem dados',ha='center',va='center'); ax.axis('off'); return
                annot = np.array([[f"{int(cm[i,j])}\n({cm_pct[i,j]:.2f}%)" for j in range(cm.shape[1])] for i in range(cm.shape[0])])
                sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=False, annot_kws={'size':12})
                ax.set_xlabel('Predito'); ax.set_ylabel('Verdadeiro')
                ax.set_xticklabels(['0','1']); ax.set_yticklabels(['0','1'])
                ax.set_title(title)

            plot_cm(axes[0], cm_train, cm_train_pct, 'Matriz de Confusão - Treino')
            plot_cm(axes[1], cm_test, cm_test_pct, 'Matriz de Confusão - Teste')

            plt.tight_layout()
            try:
                png_path = IMAGES_OUT / f'confusion_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()
    except Exception as e:
        print('Não foi possível gerar matrizes de confusão:', e)

    # -------- Importâncias médias + Pareto --------
    try:
        df_imp = pd.DataFrame(importancias_lista).replace('%','', regex=True)
        cols = [c for c in df_imp.columns if c != 'Fold']
        try:
            exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
        except Exception:
            exclude_lower = set()
        cols = [c for c in cols if c.lower() not in exclude_lower]

        if len(cols) == 0:
            print('Nenhuma feature disponível para plotar importâncias (após excluir ids).')
        else:
            df_imp_mean = df_imp[cols].astype(float).mean().sort_values(ascending=True)
            height = max(6, 0.25 * len(df_imp_mean))
            fig, ax = plt.subplots(figsize=(10, height))
            df_imp_mean.plot(kind='barh', ax=ax, color='tab:blue')
            ax.set_xlabel('Importância média (%)'); ax.set_title('Importância média das features (%)')
            plt.tight_layout(); plt.subplots_adjust(left=0.30)
            try:
                (IMAGES_OUT / f'feature_importances_{N_SPLITS}.png').write_bytes(b'')  # touch
                fig.savefig(IMAGES_OUT / f'feature_importances_{N_SPLITS}.png', dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()

            # Tabela
            try:
                table_df = df_imp_mean.sort_values(ascending=False).to_frame(name='Importance (%)').round(2)
                fig, ax = plt.subplots(figsize=(8, max(4, 0.3*len(table_df))))
                ax.axis('off')
                ax.table(cellText=table_df.values, colLabels=table_df.columns,
                         rowLabels=table_df.index, loc='center')
                ax.set_title('Tabela: Importância média das features (%)', pad=20)
                plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close()
            except Exception as e_table:
                print('Não foi possível gerar tabela de importâncias:', e_table)

            # Pareto 80/20
            try:
                imp_desc = df_imp_mean.sort_values(ascending=False)
                pareto_df = imp_desc.to_frame(name='importance_pct')
                pareto_df['cumulative_pct'] = pareto_df['importance_pct'].cumsum()
                threshold_pct = (PARETO_THRESHOLD if 'PARETO_THRESHOLD' in globals() else 0.8)*100
                import numpy as _np
                pos = _np.searchsorted(pareto_df['cumulative_pct'].values, threshold_pct)
                selected_features = pareto_df.iloc[:pos+1].index.tolist() if pos < len(pareto_df) else pareto_df.index.tolist()
                PARETO_SELECTED_FEATURES = selected_features

                # Gráfico Pareto
                fig, ax1 = plt.subplots(figsize=(10, 6))
                x = pareto_df.index.astype(str)
                ax1.bar(x, pareto_df['importance_pct'], color='tab:blue')
                ax1.set_ylabel('Importância (%)'); ax1.set_xticklabels(x, rotation=90)
                ax2 = ax1.twinx(); ax2.plot(x, pareto_df['cumulative_pct'], color='red', marker='o')
                ax2.axhline(threshold_pct, color='gray', linestyle='--'); ax2.set_ylabel('Cumulative (%)')
                ax1.set_title(f'Pareto (threshold={int(threshold_pct)}%) - selecionadas: {len(PARETO_SELECTED_FEATURES)}')
                plt.tight_layout()
                try:
                    fig.savefig(IMAGES_OUT / f'pareto_{int(threshold_pct)}.png', dpi=300, bbox_inches='tight')
                except Exception:
                    pass
                pdf.savefig(fig, bbox_inches='tight'); plt.close()
            except Exception as e_pareto:
                print('Não foi possível gerar análise Pareto:', e_pareto)
    except Exception as e:
        print('Não foi possível gerar gráfico de importâncias:', e)

    # -------- Tabela de variâncias (numéricas) --------
    try:
        if not (train_df.empty or test_df.empty):
            # Apenas colunas numéricas comuns
            feats_common = [c for c in train_df.columns if (c in test_df.columns)]
            # drop não-numéricas e colunas de meta
            drop_meta = set(['fold','orig_index','y_train','y_test','y_pred','y_proba','prob_0','prob_1'])
            feats_common = [c for c in feats_common if c not in drop_meta]
            train_num = train_df[feats_common].apply(pd.to_numeric, errors='coerce')
            test_num  = test_df[feats_common].apply(pd.to_numeric, errors='coerce')
            valid_cols = [c for c in feats_common if train_num[c].notna().any() and test_num[c].notna().any()]
            if len(valid_cols) > 0:
                var_train = train_num[valid_cols].var(ddof=1, numeric_only=True)
                var_test  = test_num[valid_cols].var(ddof=1, numeric_only=True)
                var_df = pd.DataFrame({'var_train': var_train, 'var_test': var_test})
                var_df['diff'] = var_df['var_train'] - var_df['var_test']
                var_df['pct_diff'] = (var_df['diff'].abs() / var_df['var_train'].replace({0: np.nan})) * 100
                try:
                    thresh_pct = VAR_DIFF_PCT_THRESHOLD * 100
                except Exception:
                    thresh_pct = 10.0
                var_df['high_low'] = var_df['pct_diff'].apply(lambda x: 'High' if pd.notna(x) and x >= thresh_pct else 'Low')
                var_df = var_df.sort_values('var_train', ascending=False)

                out_var_csv = CSV_OUT / 'feature_variances_train_test.csv'
                var_df.reset_index().rename(columns={'index':'feature'}).to_csv(out_var_csv, index=False)

                # Página
                fig, ax = plt.subplots(figsize=(11.69, min(12, 0.25*len(var_df)+2)))
                ax.axis('off')
                ax.table(cellText=var_df.round(4).reset_index().values,
                         colLabels=['feature','var_train','var_test','diff','pct_diff','high_low'],
                         loc='center').auto_set_font_size(False)
                ax.set_title('Tabela: Variância das Features (Treino vs Teste)')
                plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close()
            else:
                print('Nenhuma feature numérica válida em comum para variância.')
        else:
            print('Dados de treino/teste ausentes; não foi possível calcular variâncias.')
    except Exception as e_var:
        print('Erro ao calcular tabela de variâncias:', e_var)

    # -------- Violinos para features do Pareto --------
    try:
        if ('PARETO_SELECTED_FEATURES' in globals()) and PARETO_SELECTED_FEATURES and not (train_df.empty or test_df.empty):
            violin_features = [f for f in PARETO_SELECTED_FEATURES if (f in train_df.columns and f in test_df.columns)]
            if len(violin_features) > 0:
                combined = pd.concat([
                    train_df[violin_features].assign(__set__='train'),
                    test_df[violin_features].assign(__set__='test')
                ], ignore_index=True)
                per_page = 6
                import math
                pages = math.ceil(len(violin_features)/per_page)
                for p in range(pages):
                    subset = violin_features[p*per_page : (p+1)*per_page]
                    rows = 3
                    fig, axes = plt.subplots(rows, 2, figsize=(8.27, 11.69))
                    axes = axes.flatten()
                    for i, feat in enumerate(subset):
                        ax = axes[i]
                        sns.violinplot(x='__set__', y=feat, data=combined,
                                       order=['train','test'], ax=ax, inner='quartile',
                                       palette=['#4c72b0','#55a868'])
                        ax.set_title(f'Violin: {feat} (train vs test)')
                        ax.set_xlabel('')
                    for j in range(len(subset), len(axes)):
                        axes[j].axis('off')
                    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close()
            else:
                print('Nenhuma das features do Pareto está presente em train/test.')
        else:
            print('Sem PARETO_SELECTED_FEATURES ou train/test vazios; violinos omitidos.')
    except Exception as e_violin:
        print('Não foi possível gerar gráficos de violino:', e_violin)

# ===================== RECONSTRUIR train_df / test_df (se possível) =====================
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.metrics import roc_curve, auc, confusion_matrix

# Pastas auxiliares
try:
    IMAGES_OUT
except NameError:
    IMAGES_OUT = PDF_OUT.parent / "images"
IMAGES_OUT.mkdir(parents=True, exist_ok=True)

def _safe_concat(lst):
    return pd.concat(lst, ignore_index=True) if (isinstance(lst, list) and len(lst) > 0) else pd.DataFrame()

# Tenta reconstruir os dataframes globais a partir das partes do CV
X_train_global = _safe_concat(globals().get('X_train_parts', []))
y_train_global = _safe_concat(globals().get('y_train_parts', []))
y_train_pred_global = _safe_concat(globals().get('y_train_pred_parts', []))

X_test_global  = _safe_concat(globals().get('X_test_parts', []))
y_test_global  = _safe_concat(globals().get('y_test_parts', []))
y_test_pred_global  = _safe_concat(globals().get('y_test_pred_parts', []))

# Renomeia coluna "index" para "orig_index" (índice original do df)
for _df in [X_train_global, y_train_global, y_train_pred_global,
            X_test_global,  y_test_global,  y_test_pred_global]:
    if isinstance(_df, pd.DataFrame) and ('index' in _df.columns):
        _df.rename(columns={'index': 'orig_index'}, inplace=True)

# Monta train_df e test_df completos (com y_true, prob_1, y_pred)
if not X_train_global.empty and not y_train_global.empty:
    train_df = X_train_global.merge(y_train_global, on=['orig_index','fold'], how='left')
    if not y_train_pred_global.empty:
        train_df = train_df.merge(y_train_pred_global, on=['orig_index','fold'], how='left')
    if 'prob_1' in train_df.columns:
        train_df['y_proba'] = train_df['prob_1']
        train_df['y_pred'] = (train_df['y_proba'] >= THRESHOLD).astype(int)
else:
    train_df = pd.DataFrame()

if not X_test_global.empty and not y_test_global.empty:
    test_df = X_test_global.merge(y_test_global, on=['orig_index','fold'], how='left')
    if not y_test_pred_global.empty:
        test_df = test_df.merge(y_test_pred_global, on=['orig_index','fold'], how='left')
    if 'prob_1' in test_df.columns:
        test_df['y_proba'] = test_df['prob_1']
        test_df['y_pred'] = (test_df['y_proba'] >= THRESHOLD).astype(int)
else:
    test_df = pd.DataFrame()

# ===================== AJUSTE 8/1/1 (mínimo) =====================
# Teste = apenas fold 10; Treino = folds 1..9 (8+1)
if isinstance(test_df, pd.DataFrame) and (not test_df.empty) and ('fold' in test_df.columns):
    test_df = test_df[test_df['fold'] == 10].copy()

if isinstance(train_df, pd.DataFrame) and (not train_df.empty) and ('fold' in train_df.columns):
    train_df = train_df[train_df['fold'] != 10].copy()

# ===================== PDF RESUMO XGBOOST =====================
with PdfPages(PDF_OUT / f'XGB_CV_SUMMARY_{N_SPLITS}.pdf') as pdf:
    # -------- Página de parâmetros --------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    parametros = xgb_params.copy()
    parametros['threshold'] = THRESHOLD
    parametros['n_splits'] = N_SPLITS
    texto = 'Algoritmo: XGBoost\n' + '\n'.join([f'{k}: {v}' for k, v in parametros.items()])
    ax.text(0, 1, texto, fontsize=11, family='monospace', va='top')
    ax.set_title('📌 Parâmetros do Modelo - XGBoost')
    try:
        png_path = IMAGES_OUT / f'params_{N_SPLITS}.png'
        fig.savefig(png_path, dpi=300, bbox_inches='tight')
    except Exception:
        pass
    pdf.savefig(fig); plt.close()

    # -------- Página de métricas resumidas --------
    if 'df_resultados' in globals() and not df_resultados.empty:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.axis('off')
        display_df = df_resultados.groupby(['Conjunto']).mean(numeric_only=True).T.round(4)
        table = ax.table(cellText=display_df.values,
                         colLabels=display_df.columns,
                         rowLabels=display_df.index,
                         loc='center')
        table.auto_set_font_size(False); table.set_fontsize(9)
        try:
            png_path = IMAGES_OUT / f'metrics_summary_{N_SPLITS}.png'
            fig.savefig(png_path, dpi=300, bbox_inches='tight')
        except Exception:
            pass
        pdf.savefig(fig); plt.close()

    # -------- ROC - Treino vs Teste --------
    try:
        plotted = False
        # Preferência: usar dataframes globais (já filtrados para 8/1/1)
        if ('y_train' in train_df.columns) and ('y_proba' in train_df.columns) and (not train_df.empty):
            y_train_all = train_df['y_train']; y_score_train_all = train_df['y_proba']
            fpr_train, tpr_train, _ = roc_curve(y_train_all, y_score_train_all)
            roc_auc_train = auc(fpr_train, tpr_train); plotted = True
        elif 'fprs_train' in globals() and len(fprs_train) > 0:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = [np.interp(mean_fpr, fpr, tpr) for fpr, tpr in zip(fprs_train, tprs_train)]
            for t in tprs: t[0] = 0.0
            mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
            fpr_train, tpr_train = mean_fpr, mean_tpr
            roc_auc_train = np.mean(aurocs_train) if 'aurocs_train' in globals() and len(aurocs_train)>0 else auc(fpr_train,tpr_train)
            plotted = True

        if ('y_test' in test_df.columns) and ('y_proba' in test_df.columns) and (not test_df.empty):
            y_test_all = test_df['y_test']; y_score_test_all = test_df['y_proba']
            fpr_test, tpr_test, _ = roc_curve(y_test_all, y_score_test_all)
            roc_auc_test = auc(fpr_test, tpr_test); plotted = True
        elif 'fprs_test' in globals() and len(fprs_test) > 0:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = [np.interp(mean_fpr, fpr, tpr) for fpr, tpr in zip(fprs_test, tprs_test)]
            for t in tprs: t[0] = 0.0
            mean_tpr = np.mean(tprs, axis=0); mean_tpr[-1] = 1.0
            fpr_test, tpr_test = mean_fpr, mean_tpr
            roc_auc_test = np.mean(aurocs_test) if 'aurocs_test' in globals() and len(aurocs_test)>0 else auc(fpr_test,tpr_test)
            plotted = True

        if plotted:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            try:
                axes[0].plot(fpr_train, tpr_train, label=f'AUC = {roc_auc_train:.4f}')
                axes[0].plot([0,1],[0,1],'k--',lw=1)
                axes[0].set_title('ROC - Treino'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
                axes[0].legend(loc='lower right')
            except Exception:
                axes[0].text(0.5,0.5,'Sem dados de treino',ha='center'); axes[0].axis('off')
            try:
                axes[1].plot(fpr_test, tpr_test, label=f'AUC = {roc_auc_test:.4f}')
                axes[1].plot([0,1],[0,1],'k--',lw=1)
                axes[1].set_title('ROC - Teste'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
                axes[1].legend(loc='lower right')
            except Exception:
                axes[1].text(0.5,0.5,'Sem dados de teste',ha='center'); axes[1].axis('off')
            plt.tight_layout()
            try:
                png_path = IMAGES_OUT / f'roc_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()
        else:
            print('Não foi possível gerar curvas ROC (dados ausentes)')
    except Exception as e:
        print('Não foi possível gerar ROC agregado:', e)

    # -------- Matrizes de Confusão --------
    try:
        def build_cm_from_df(df_block, true_col, pred_col):
            y_true = df_block[true_col]; y_pred = df_block[pred_col]
            cm = confusion_matrix(y_true, y_pred)
            total = cm.sum(); cm_percent = cm/total*100 if total>0 else np.zeros_like(cm, dtype=float)
            return cm, cm_percent

        cm_train = cm_train_pct = cm_test = cm_test_pct = None

        if ('y_train' in train_df.columns) and ('y_pred' in train_df.columns) and (not train_df.empty):
            cm_train, cm_train_pct = build_cm_from_df(train_df, 'y_train', 'y_pred')
        elif ('df_resultados' in globals()) and ('TP' in df_resultados.columns):
            tr = df_resultados[df_resultados['Conjunto']=='Treino']
            TP,FP,TN,FN = int(tr['TP'].sum()), int(tr['FP'].sum()), int(tr['TN'].sum()), int(tr['FN'].sum())
            cm_train = np.array([[TN, FP],[FN, TP]])
            total = cm_train.sum(); cm_train_pct = cm_train/total*100 if total>0 else np.zeros_like(cm_train,float)

        if ('y_test' in test_df.columns) and ('y_pred' in test_df.columns) and (not test_df.empty):
            cm_test, cm_test_pct = build_cm_from_df(test_df, 'y_test', 'y_pred')
        elif ('df_resultados' in globals()) and ('TP' in df_resultados.columns):
            ts = df_resultados[df_resultados['Conjunto']=='Teste']
            TP,FP,TN,FN = int(ts['TP'].sum()), int(ts['FP'].sum()), int(ts['TN'].sum()), int(ts['FN'].sum())
            cm_test = np.array([[TN, FP],[FN, TP]])
            total = cm_test.sum(); cm_test_pct = cm_test/total*100 if total>0 else np.zeros_like(cm_test,float)

        if (cm_train is None and cm_test is None):
            print('Não há dados suficientes para gerar matrizes de confusão.')
        else:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            def plot_cm(ax, cm, cm_pct, title):
                if cm is None:
                    ax.text(0.5,0.5,'Sem dados',ha='center',va='center'); ax.axis('off'); return
                annot = np.array([[f"{int(cm[i,j])}\n({cm_pct[i,j]:.2f}%)" for j in range(cm.shape[1])] for i in range(cm.shape[0])])
                sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=False, annot_kws={'size':12})
                ax.set_xlabel('Predito'); ax.set_ylabel('Verdadeiro')
                ax.set_xticklabels(['0','1']); ax.set_yticklabels(['0','1'])
                ax.set_title(title)

            plot_cm(axes[0], cm_train, cm_train_pct, 'Matriz de Confusão - Treino')
            plot_cm(axes[1], cm_test, cm_test_pct, 'Matriz de Confusão - Teste')

            plt.tight_layout()
            try:
                png_path = IMAGES_OUT / f'confusion_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()
    except Exception as e:
        print('Não foi possível gerar matrizes de confusão:', e)

    # -------- Importâncias médias + Pareto --------
    try:
        df_imp = pd.DataFrame(importancias_lista).replace('%','', regex=True)
        cols = [c for c in df_imp.columns if c != 'Fold']
        try:
            exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
        except Exception:
            exclude_lower = set()
        cols = [c for c in cols if c.lower() not in exclude_lower]

        if len(cols) == 0:
            print('Nenhuma feature disponível para plotar importâncias (após excluir ids).')
        else:
            df_imp_mean = df_imp[cols].astype(float).mean().sort_values(ascending=True)
            height = max(6, 0.25 * len(df_imp_mean))
            fig, ax = plt.subplots(figsize=(10, height))
            df_imp_mean.plot(kind='barh', ax=ax, color='tab:blue')
            ax.set_xlabel('Importância média (%)'); ax.set_title('Importância média das features (%)')
            plt.tight_layout(); plt.subplots_adjust(left=0.30)
            try:
                (IMAGES_OUT / f'feature_importances_{N_SPLITS}.png').write_bytes(b'')  # touch
                fig.savefig(IMAGES_OUT / f'feature_importances_{N_SPLITS}.png', dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()

            # Tabela
            try:
                table_df = df_imp_mean.sort_values(ascending=False).to_frame(name='Importance (%)').round(2)
                fig, ax = plt.subplots(figsize=(8, max(4, 0.3*len(table_df))))
                ax.axis('off')
                ax.table(cellText=table_df.values, colLabels=table_df.columns,
                         rowLabels=table_df.index, loc='center')
                ax.set_title('Tabela: Importância média das features (%)', pad=20)
                plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close()
            except Exception as e_table:
                print('Não foi possível gerar tabela de importâncias:', e_table)

            # Pareto 80/20
            try:
                imp_desc = df_imp_mean.sort_values(ascending=False)
                pareto_df = imp_desc.to_frame(name='importance_pct')
                pareto_df['cumulative_pct'] = pareto_df['importance_pct'].cumsum()
                threshold_pct = (PARETO_THRESHOLD if 'PARETO_THRESHOLD' in globals() else 0.8)*100
                import numpy as _np
                pos = _np.searchsorted(pareto_df['cumulative_pct'].values, threshold_pct)
                selected_features = pareto_df.iloc[:pos+1].index.tolist() if pos < len(pareto_df) else pareto_df.index.tolist()
                PARETO_SELECTED_FEATURES = selected_features

                # Gráfico Pareto
                fig, ax1 = plt.subplots(figsize=(10, 6))
                x = pareto_df.index.astype(str)
                ax1.bar(x, pareto_df['importance_pct'], color='tab:blue')
                ax1.set_ylabel('Importância (%)'); ax1.set_xticklabels(x, rotation=90)
                ax2 = ax1.twinx(); ax2.plot(x, pareto_df['cumulative_pct'], color='red', marker='o')
                ax2.axhline(threshold_pct, color='gray', linestyle='--'); ax2.set_ylabel('Cumulative (%)')
                ax1.set_title(f'Pareto (threshold={int(threshold_pct)}%) - selecionadas: {len(PARETO_SELECTED_FEATURES)}')
                plt.tight_layout()
                try:
                    fig.savefig(IMAGES_OUT / f'pareto_{int(threshold_pct)}.png', dpi=300, bbox_inches='tight')
                except Exception:
                    pass
                pdf.savefig(fig, bbox_inches='tight'); plt.close()
            except Exception as e_pareto:
                print('Não foi possível gerar análise Pareto:', e_pareto)
    except Exception as e:
        print('Não foi possível gerar gráfico de importâncias:', e)

    # -------- Tabela de variâncias (numéricas) --------
    try:
        if not (train_df.empty or test_df.empty):
            # Apenas colunas numéricas comuns
            feats_common = [c for c in train_df.columns if (c in test_df.columns)]
            # drop não-numéricas e colunas de meta
            drop_meta = set(['fold','orig_index','y_train','y_test','y_pred','y_proba','prob_0','prob_1'])
            feats_common = [c for c in feats_common if c not in drop_meta]
            train_num = train_df[feats_common].apply(pd.to_numeric, errors='coerce')
            test_num  = test_df[feats_common].apply(pd.to_numeric, errors='coerce')
            valid_cols = [c for c in feats_common if train_num[c].notna().any() and test_num[c].notna().any()]
            if len(valid_cols) > 0:
                var_train = train_num[valid_cols].var(ddof=1, numeric_only=True)
                var_test  = test_num[valid_cols].var(ddof=1, numeric_only=True)
                var_df = pd.DataFrame({'var_train': var_train, 'var_test': var_test})
                var_df['diff'] = var_df['var_train'] - var_df['var_test']
                var_df['pct_diff'] = (var_df['diff'].abs() / var_df['var_train'].replace({0: np.nan})) * 100
                try:
                    thresh_pct = VAR_DIFF_PCT_THRESHOLD * 100
                except Exception:
                    thresh_pct = 10.0
                var_df['high_low'] = var_df['pct_diff'].apply(lambda x: 'High' if pd.notna(x) and x >= thresh_pct else 'Low')
                var_df = var_df.sort_values('var_train', ascending=False)

                out_var_csv = CSV_OUT / 'feature_variances_train_test.csv'
                var_df.reset_index().rename(columns={'index':'feature'}).to_csv(out_var_csv, index=False)

                # Página
                fig, ax = plt.subplots(figsize=(11.69, min(12, 0.25*len(var_df)+2)))
                ax.axis('off')
                ax.table(cellText=var_df.round(4).reset_index().values,
                         colLabels=['feature','var_train','var_test','diff','pct_diff','high_low'],
                         loc='center').auto_set_font_size(False)
                ax.set_title('Tabela: Variância das Features (Treino vs Teste)')
                plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close()
            else:
                print('Nenhuma feature numérica válida em comum para variância.')
        else:
            print('Dados de treino/teste ausentes; não foi possível calcular variâncias.')
    except Exception as e_var:
        print('Erro ao calcular tabela de variâncias:', e_var)

    # -------- Violinos para features do Pareto --------
    try:
        if ('PARETO_SELECTED_FEATURES' in globals()) and PARETO_SELECTED_FEATURES and not (train_df.empty or test_df.empty):
            violin_features = [f for f in PARETO_SELECTED_FEATURES if (f in train_df.columns and f in test_df.columns)]
            if len(violin_features) > 0:
                combined = pd.concat([
                    train_df[violin_features].assign(__set__='train'),
                    test_df[violin_features].assign(__set__='test')
                ], ignore_index=True)
                per_page = 6
                import math
                pages = math.ceil(len(violin_features)/per_page)
                for p in range(pages):
                    subset = violin_features[p*per_page : (p+1)*per_page]
                    rows = 3
                    fig, axes = plt.subplots(rows, 2, figsize=(8.27, 11.69))
                    axes = axes.flatten()
                    for i, feat in enumerate(subset):
                        ax = axes[i]
                        sns.violinplot(x='__set__', y=feat, data=combined,
                                       order=['train','test'], ax=ax, inner='quartile',
                                       palette=['#4c72b0','#55a868'])
                        ax.set_title(f'Violin: {feat} (train vs test)')
                        ax.set_xlabel('')
                    for j in range(len(subset), len(axes)):
                        axes[j].axis('off')
                    plt.tight_layout(); pdf.savefig(fig, bbox_inches='tight'); plt.close()
            else:
                print('Nenhuma das features do Pareto está presente em train/test.')
        else:
            print('Sem PARETO_SELECTED_FEATURES ou train/test vazios; violinos omitidos.')
    except Exception as e_violin:
        print('Não foi possível gerar gráficos de violino:', e_violin)

print('PDF resumo gerado em:', PDF_OUT)


In [ ]:
# === XGBoost 8/1/1 Básico (single modelo: treina 9 folds, testa 1) ===
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier

# -------- Parâmetros (defaults se não existirem) --------
try:
    RT_RANDOM_STATE
except NameError:
    RT_RANDOM_STATE = 42

# xgb_params deve existir (como no seu código original)
# Exemplo:
# xgb_params = {
#     'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 5,
#     'subsample': 0.8, 'colsample_bytree': 0.8,
#     'random_state': 42, 'n_jobs': -1, 'eval_metric': 'logloss'
# }

# Verificações
if 'FEATURES' not in globals():
    raise RuntimeError('❌ FEATURES não encontrado — execute as células anteriores para preparar os dados')
if 'df' not in globals():
    raise RuntimeError('❌ DataFrame df não encontrado — execute as células anteriores para carregar os dados')

# -------- Cópia base --------
X_raw = df[FEATURES].copy()
y = df['y'].copy()

# Detecta categóricas
cat_cols = X_raw.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = [c for c in X_raw.columns if c not in cat_cols]

# -------- Define 8/1/1 via K-Fold (10 folds) --------
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=RT_RANDOM_STATE)
fold_indices = list(kf.split(X_raw, y))

# Regra determinística: último = TESTE, penúltimo = VAL, demais 8 = TREINO
(train_val_idx, test_idx) = fold_indices[-1]
(train_inner_idx, val_idx) = fold_indices[-2]

train8_idx = train_inner_idx
val1_idx   = val_idx
test1_idx  = test_idx

# Conjuntos efetivos
X_train8, y_train8 = X_raw.iloc[train8_idx], y.iloc[train8_idx]
X_val1,   y_val1   = X_raw.iloc[val1_idx],   y.iloc[val1_idx]
X_test1,  y_test1  = X_raw.iloc[test1_idx],  y.iloc[test1_idx]

# Treino final 9-fold = 8 + 1 validação
X_train9 = pd.concat([X_train8, X_val1], axis=0)
y_train9 = pd.concat([y_train8, y_val1], axis=0)

# -------- Pré-processamento SEM vazamento --------
# 1) OrdinalEncoder nas categóricas (fit no treino9)
if len(cat_cols) > 0:
    print(f"🔤 Codificando variáveis categóricas (fit no 9-fold): {cat_cols}")
    enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    X_train9_cat = enc.fit_transform(X_train9[cat_cols].astype('object'))
    X_test1_cat  = enc.transform(X_test1[cat_cols].astype('object'))
    X_train9_cat = pd.DataFrame(X_train9_cat, columns=cat_cols, index=X_train9.index)
    X_test1_cat  = pd.DataFrame(X_test1_cat,  columns=cat_cols, index=X_test1.index)
else:
    enc = None
    X_train9_cat = pd.DataFrame(index=X_train9.index)
    X_test1_cat  = pd.DataFrame(index=X_test1.index)

# 2) Numéricas: coerção + imputação (mediana do treino9)
if len(num_cols) > 0:
    X_train9_num = X_train9[num_cols].apply(pd.to_numeric, errors='coerce')
    X_test1_num  = X_test1[num_cols].apply(pd.to_numeric, errors='coerce')
    med = X_train9_num.median(numeric_only=True)
    X_train9_num = X_train9_num.fillna(med)
    X_test1_num  = X_test1_num.fillna(med)
else:
    X_train9_num = pd.DataFrame(index=X_train9.index)
    X_test1_num  = pd.DataFrame(index=X_test1.index)

# 3) Reconstrói na mesma ordem de FEATURES
X_train9_proc = pd.concat([X_train9_num, X_train9_cat], axis=1)[FEATURES]
X_test1_proc  = pd.concat([X_test1_num,  X_test1_cat],  axis=1)[FEATURES]

print(f"✅ X pronto (9-fold train / 1-fold test). Shapes: train={X_train9_proc.shape}, test={X_test1_proc.shape}")

# -------- Treina modelo (9-fold) --------
model_basic = XGBClassifier(**xgb_params)
model_basic.fit(X_train9_proc, y_train9)

# -------- Predições --------
y_pred_train_bin = model_basic.predict(X_train9_proc)
y_pred_test_bin  = model_basic.predict(X_test1_proc)
y_proba_train    = model_basic.predict_proba(X_train9_proc)[:, 1]
y_proba_test     = model_basic.predict_proba(X_test1_proc)[:, 1]

# -------- Métricas --------
acc_train = accuracy_score(y_train9, y_pred_train_bin)
acc_test  = accuracy_score(y_test1,  y_pred_test_bin)
f1_tr     = f1_score(y_train9, y_pred_train_bin)
f1_ts     = f1_score(y_test1,  y_pred_test_bin)
prec_tr   = precision_score(y_train9, y_pred_train_bin)
prec_ts   = precision_score(y_test1,  y_pred_test_bin)
rec_tr    = recall_score(y_train9, y_pred_train_bin)
rec_ts    = recall_score(y_test1,  y_pred_test_bin)
auc_tr    = roc_auc_score(y_train9, y_proba_train)
auc_ts    = roc_auc_score(y_test1,  y_proba_test)

print(f"✅ Acurácia treino(9 folds): {acc_train:.4f} — teste(fold10): {acc_test:.4f}")

# -------- Salvamentos (mantendo padrão de nomes) --------
# Estima TEST_SIZE efetivo do fold de teste para manter sufixo nos arquivos
TEST_SIZE_EFETIVO = len(test1_idx) / len(X_raw)
try:
    MODEL_DIR
except NameError:
    from pathlib import Path
    MODEL_DIR = Path("../model_reports/xgboost/models")
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
try:
    BASICO_CSV
except NameError:
    BASICO_CSV = Path("../model_reports/xgboost/basico/csv")
    BASICO_CSV.mkdir(parents=True, exist_ok=True)
try:
    DATASET_NAME
except NameError:
    DATASET_NAME = "dataset"

model_name = MODEL_DIR / f'xgb_basic_ts{int(TEST_SIZE_EFETIVO*100)}_rs{RT_RANDOM_STATE}.pkl'
joblib.dump(model_basic, model_name)

# Salva as matrizes usadas (9-fold train / 1-fold test)
X_train9_proc.to_csv(BASICO_CSV / f'X_train_xgb_basic_ts{int(TEST_SIZE_EFETIVO*100)}_rs{RT_RANDOM_STATE}.csv', index=False)
X_test1_proc.to_csv( BASICO_CSV / f'X_test_xgb_basic_ts{int(TEST_SIZE_EFETIVO*100)}_rs{RT_RANDOM_STATE}.csv',  index=False)
y_train9.to_csv(    BASICO_CSV / f'y_train_xgb_basic_ts{int(TEST_SIZE_EFETIVO*100)}_rs{RT_RANDOM_STATE}.csv', index=True)
y_test1.to_csv(     BASICO_CSV / f'y_test_xgb_basic_ts{int(TEST_SIZE_EFETIVO*100)}_rs{RT_RANDOM_STATE}.csv',  index=True)
joblib.dump(FEATURES, MODEL_DIR / f'features_xgb_basic_ts{int(TEST_SIZE_EFETIVO*100)}_rs{RT_RANDOM_STATE}.pkl')

# Tabela de métricas (mantida)
metrics_basic = pd.DataFrame([{
    'model': str(model_name.name),
    'test_size': TEST_SIZE_EFETIVO,
    'random_state': RT_RANDOM_STATE,
    'acc_train': acc_train,
    'acc_test':  acc_test,
    'f1_train':  f1_tr,
    'f1_test':   f1_ts,
    'precision_train': prec_tr,
    'precision_test':  prec_ts,
    'recall_train':    rec_tr,
    'recall_test':     rec_ts,
    'auc_train': auc_tr,
    'auc_test':  auc_ts
}])
metrics_basic.to_csv(BASICO_CSV / 'xgb_model_basic.csv', index=False)

# Dataset augmentado (somente com previsões do TESTE = fold 10)
df_aug = df.copy()
df_aug['y_pred']  = np.nan
df_aug['y_proba'] = np.nan
df_aug.loc[X_test1_proc.index, 'y_pred']  = y_pred_test_bin
df_aug.loc[X_test1_proc.index, 'y_proba'] = y_proba_test
df_aug.to_csv(
    BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_xgb_basic_ts{int(TEST_SIZE_EFETIVO*100)}_rs{RT_RANDOM_STATE}.csv',
    index=False
)

# Amostras aleatórias para explicabilidade (a partir do teste = fold 10)
num_instancias = 12
np.random.seed(42)
indices_aleatorios = np.random.choice(X_test1_proc.index, size=min(num_instancias, len(X_test1_proc)), replace=False)

print('📦 Modelo salvo em:', model_name)
print('📊 Métricas salvas em:', BASICO_CSV / 'xgb_model_basic.csv')
print('🧩 Dataset augmentado salvo em:', BASICO_CSV / f'database_used_{DATASET_NAME}_with_preds_xgb_basic_ts{int(TEST_SIZE_EFETIVO*100)}_rs{RT_RANDOM_STATE}.csv')
print('🎯 Índices amostra aleatórios (teste/fold10):', indices_aleatorios)


In [ ]:
# ==============================================
# Salvar X_test_basic_full.csv e X_train_basic_full.csv
# com todas as colunas do arquivo cru + y, y_pred, y_proba
# Compatível com 8/1/1 e split simples
# ==============================================

import pandas as pd
import numpy as np
from pathlib import Path

# -------- Paths de saída (mantém nomes originais) --------
try:
    BASICO_CSV
except NameError:
    # fallback genérico se não existir
    BASICO_CSV = Path("../model_reports/basico/csv")
BASICO_CSV.mkdir(parents=True, exist_ok=True)

# -------- Carrega dataset cru --------
raw_df = pd.read_csv(DATASET_PATH)

# Target (mantém se já existir)
if 'y' not in raw_df.columns:
    if 'stroke' in raw_df.columns:
        raw_df['y'] = raw_df['stroke']
    else:
        raise ValueError("❌ Nenhuma coluna alvo encontrada no dataset (esperado 'stroke' ou 'y').")

# -------- Auxiliares --------
def _pick_first_existing(var_names):
    """Retorna o primeiro objeto existente no escopo global a partir da lista de nomes."""
    for name in var_names:
        if name in globals():
            return globals()[name]
    return None

def _pick_model():
    """Tenta achar um modelo treinado (em ordem de preferência)."""
    return _pick_first_existing([
        'clf_final',     # usado em 8/1/1 XGB/RF final
        'model_basic',   # usado no básico
        'model',         # genérico
    ])

def _safe_proba_from_model(model, Xmat):
    """Tenta obter predict_proba, senão cai num fallback binário (0/1)."""
    if model is not None and hasattr(model, 'predict_proba'):
        try:
            proba = model.predict_proba(Xmat)
            if isinstance(proba, np.ndarray) and proba.ndim == 2 and proba.shape[1] >= 2:
                return proba[:, 1]
        except Exception:
            pass
    # fallback: se não houver predict_proba, tenta com predict
    try:
        yhat = model.predict(Xmat) if model is not None else np.zeros(len(Xmat), dtype=int)
    except Exception:
        yhat = np.zeros(len(Xmat), dtype=int)
    return np.where(yhat == 1, 1.0, 0.0)

def _ensure_series(values, index, name):
    """Normaliza para Series alinhada ao índice."""
    s = pd.Series(values, index=index, name=name)
    return s

def save_full_split(indices, y_true, y_pred, y_proba, split_name):
    """Monta um DataFrame completo com o conjunto original + y_true, y_pred, y_proba e salva."""
    df_full = raw_df.loc[indices].copy()
    df_full['y']       = _ensure_series(y_true,  indices, 'y')
    df_full['y_pred']  = _ensure_series(y_pred,  indices, 'y_pred')
    df_full['y_proba'] = _ensure_series(y_proba, indices, 'y_proba')
    out_path = BASICO_CSV / f"X_{split_name}_basic_full.csv"
    df_full.to_csv(out_path, index=True)
    print(f"✅ Arquivo salvo: {out_path} (linhas: {len(df_full)})")

# -------- Detecta contexto 8/1/1 ou split simples --------
# 8/1/1 (preferido se existir)
X_train_ctx = _pick_first_existing(['X_train9_proc', 'X_train'])   # 9 folds proc OU train simples
X_test_ctx  = _pick_first_existing(['X_test1_proc', 'X_test'])     # fold 10 proc OU test simples
y_train_ctx = _pick_first_existing(['y_train9', 'y_train'])
y_test_ctx  = _pick_first_existing(['y_test1',  'y_test'])

# Se nada foi encontrado, não há como proceder
if X_train_ctx is None or X_test_ctx is None or y_train_ctx is None or y_test_ctx is None:
    raise RuntimeError("❌ Variáveis de treino/teste não encontradas. Execute antes o treino (8/1/1 ou split simples).")

# -------- Probabilidades / Predições: usa variáveis já existentes se houver; senão, calcula --------
# Preferência por variáveis já criadas no pipeline
y_proba_test = _pick_first_existing(['y_proba_test', 'y_proba_test_C1', 'y_proba_test_all'])  # várias convenções
y_proba_trn  = _pick_first_existing(['y_proba_train', 'y_proba_train_C1', 'y_proba_train_all'])

y_pred_test_bin = _pick_first_existing(['y_pred_test_bin'])
y_pred_trn_bin  = _pick_first_existing(['y_pred_train_bin'])

model_used = _pick_model()

# Teste
if y_proba_test is None:
    y_proba_test = _safe_proba_from_model(model_used, X_test_ctx)
if y_pred_test_bin is None:
    try:
        y_pred_test_bin = model_used.predict(X_test_ctx)
    except Exception:
        y_pred_test_bin = (np.asarray(y_proba_test) >= 0.5).astype(int)

# Treino
if y_proba_trn is None:
    y_proba_trn = _safe_proba_from_model(model_used, X_train_ctx)
if y_pred_trn_bin is None:
    try:
        y_pred_trn_bin = model_used.predict(X_train_ctx)
    except Exception:
        y_pred_trn_bin = (np.asarray(y_proba_trn) >= 0.5).astype(int)

# -------- Salva arquivos (mantém nomes originais) --------
save_full_split(
    indices=X_test_ctx.index,
    y_true=y_test_ctx,
    y_pred=y_pred_test_bin,
    y_proba=y_proba_test,
    split_name="test"
)

save_full_split(
    indices=X_train_ctx.index,
    y_true=y_train_ctx,
    y_pred=y_pred_trn_bin,
    y_proba=y_proba_trn,
    split_name="train"
)
